# Phase 01.00 — Environment & preflight

This notebook verifies the fixed project root, the selected Python kernel, compiler/runtime prerequisites, RTX 3090/BF16 availability, required packages, and the pinned Vintern contract. It deliberately keeps FlashAttention off.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


## 1. Kernel and system checks

The expected interpreter is `/root/venvs/roadbuddy-rtx3090-py310/bin/python`. A different executable means the notebook is attached to the wrong kernel.


In [2]:
import platform, shutil, subprocess, sys, sysconfig
import torch

EXPECTED_PYTHON = Path("/root/venvs/roadbuddy-rtx3090-py310/bin/python")
checks = {
    "python": sys.executable,
    "python_version": platform.python_version(),
    "expected_kernel": Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(),
    "gcc": shutil.which("gcc"),
    "g++": shutil.which("g++"),
    "Python.h": str(Path(sysconfig.get_paths()["include"]) / "Python.h"),
    "Python.h_exists": (Path(sysconfig.get_paths()["include"]) / "Python.h").is_file(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda,
}
if torch.cuda.is_available():
    checks.update({"gpu": torch.cuda.get_device_name(0), "capability": torch.cuda.get_device_capability(0), "bf16": torch.cuda.is_bf16_supported()})
checks


{'python': '/root/venvs/roadbuddy-rtx3090-py310/bin/python',
 'python_version': '3.10.12',
 'expected_kernel': True,
 'gcc': '/usr/bin/gcc',
 'g++': '/usr/bin/g++',
 'Python.h': '/usr/include/python3.10/Python.h',
 'Python.h_exists': True,
 'torch': '2.13.0+cu126',
 'cuda_available': True,
 'cuda_runtime': '12.6',
 'gpu': 'NVIDIA GeForce RTX 3090',
 'capability': (8, 6),
 'bf16': True}

In [3]:
assert checks["expected_kernel"], f"Wrong kernel: {sys.executable}"
assert checks["gcc"] and checks["g++"], "gcc/g++ missing; install build-essential"
assert checks["Python.h_exists"], "Python development headers missing; install python3.10-dev"
assert torch.cuda.is_available(), "CUDA is unavailable"
assert "RTX 3090" in torch.cuda.get_device_name(0), f"Expected RTX 3090, got {torch.cuda.get_device_name(0)}"
assert torch.cuda.is_bf16_supported(), "BF16 is required by this notebook set"
print("System checks: PASS")


System checks: PASS


## 2. Package imports


In [4]:
import importlib.metadata as metadata

required = ["transformers", "peft", "accelerate", "torchvision", "pandas", "scikit-learn", "opencv-python-headless", "ipykernel"]
versions = {}
for package in required:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "MISSING"
versions


{'transformers': '4.57.6',
 'peft': '0.19.1',
 'accelerate': '1.14.0',
 'torchvision': '0.28.0+cu126',
 'pandas': '2.3.3',
 'scikit-learn': '1.7.2',
 'opencv-python-headless': '4.10.0.84',
 'ipykernel': '7.3.0'}

In [5]:
missing = [name for name, version in versions.items() if version == "MISSING"]
assert not missing, f"Missing packages: {missing}"
print("Package checks: PASS")


Package checks: PASS


## 3. Pinned Vintern contract

This is a real model-load check. It confirms the revision exposes `model.num_image_token`, uses the native `Hermes-2` template, and was loaded with eager attention.


In [6]:
RUN_MODEL_LOAD = True

if RUN_MODEL_LOAD:
    model, tokenizer = load_model_and_tokenizer(training=False)
    model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    model_contract = {
        "model_id": MODEL_ID,
        "revision": MODEL_REVISION,
        "class": f"{model.__class__.__module__}.{model.__class__.__name__}",
        "num_image_token": int(model.num_image_token),
        "template": model.template,
        "img_context_token_id": model.img_context_token_id,
        "flash_attention": False,
    }
    assert model.img_context_token_id != tokenizer.unk_token_id
    save_json(PATHS.phase1_output / "preflight.json", {**checks, "packages": versions, "model": model_contract})
    display(model_contract)
    del model
    torch.cuda.empty_cache()
    print("Pinned model contract: PASS")


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


{'model_id': '5CD-AI/Vintern-1B-v3_5',
 'revision': 'b98f263eab246eb5269ade64edbdca8a887dc44d',
 'class': 'transformers_modules._5CD_hyphen_AI.Vintern_hyphen_1B_hyphen_v3_5.b98f263eab246eb5269ade64edbdca8a887dc44d.modeling_internvl_chat.InternVLChatModel',
 'num_image_token': 256,
 'template': 'Hermes-2',
 'img_context_token_id': 151667,
 'flash_attention': False}

Pinned model contract: PASS


## Gate

Proceed only when every cell above passes. The next notebook creates or validates the frozen Phase 01 split.


## Nhận xét sau lần chạy Phase 01.00

**Trạng thái:** PASS. Notebook đã xác nhận kernel và model contract hoạt động trên đúng môi trường RoadBuddy.

### Kết quả chính

- Python executable: `/root/venvs/roadbuddy-rtx3090-py310/bin/python`, Python 3.10.12.
- PyTorch `2.13.0+cu126`; CUDA khả dụng trên NVIDIA GeForce RTX 3090; BF16 được hỗ trợ.
- `ipykernel 7.3.0` và `opencv-python-headless 4.10.0.84` import thành công.
- Model `5CD-AI/Vintern-1B-v3_5` được khóa tại revision `b98f263eab246eb5269ade64edbdca8a887dc44d` và tải thành công.
- Remote model class, template `Hermes-2`, `num_image_token=256` và image-context token đều đạt contract. FlashAttention được tắt có chủ đích.

### Nhận định

Môi trường đủ điều kiện cho toàn bộ Phase01 và kết quả có khả năng tái lập ở mức model revision, seed và kernel. Việc khóa revision là quan trọng vì model dùng `trust_remote_code=True`; thay revision có thể làm thay đổi API `chat`, template hoặc token alignment.

### Cảnh báo và hành động

- Thiếu `ipywidgets` chỉ làm mất progress bar tương tác, không ảnh hưởng kết quả.
- FlashAttention2 chưa cài nhưng pipeline dùng eager attention, vì vậy đây không phải lỗi.
- Cảnh báo deprecated từ `timm` nằm trong remote code của model; không nên sửa trực tiếp cache model.
- Khi chuyển server/GPU, cần chạy lại notebook này trước các phase sau để xác nhận CUDA, BF16 và model contract.